# NB01 · Bronze (Spain Utilities Pulse)

Descarga **incremental** de las cuatro fuentes a `Files/bronze/<fuente>/`, tal como llegan (JSON).

| Fuente | Carpeta Bronze | Datos | Desde |
|---|---|---|---|
| REE REData | `ree_demanda` | Demanda eléctrica peninsular diaria | 2024-01-01 |
| ENTSOG | `entsog_flujo_fisico` | Flujo físico diario de Enagás por punto y dirección | 2024-01-01 |
| GIE AGSI+ | `agsi_almacenamiento` | Almacenamiento subterráneo de gas de España | 2024-01-01 |
| MITECO | `miteco_embalses` | Reserva semanal por embalse (base Access `BD-Embalses`) | 2014-01-01 |

**Cómo funciona la carga incremental**
- **Primera ejecución:** baja todo desde la fecha de inicio de cada fuente.
- **Siguientes:** pide desde `última fecha guardada - SOLAPE_DIAS` (7 días; 56 en MITECO) para recoger las correcciones que publican las fuentes.
- **Bronze no se sobrescribe:** cada ejecución escribe un JSON nuevo por fuente con `{meta, data}`. La última fecha guardada va en el nombre del archivo (`..._hasta20261122.json`), así no hace falta abrir los JSON para saber dónde se quedó la carga.
- Si una fuente no trae datos nuevos, no se escribe archivo.
- La deduplicación y el `MERGE` ocurren en Silver (NB02), no aquí.

**Agua (MITECO).** El zip trae una base Access con el histórico desde 1988; se lee con la librería `access_parser` (primera celda). Se guarda desde 2014 porque Gold necesita 10 años de historia para calcular la media de 10 años. El solape es de 8 semanas porque MITECO publica datos provisionales que revisa después.

**Control de errores.** La última celda lanza un error si alguna fuente falla o se omite, para que el pipeline marque la actividad como fallida y no ejecute Silver ni Gold.

AGSI+ necesita la clave en `Files/config/utilities_config.json` → `{"agsi_key": "TU_CLAVE"}`.

**Resultado de la ejecución del 25 nov 2026**

| Fuente | Modo | Desde | Filas | Último dato |
|---|---|---|---|---|
| `ree_demanda` | incremental | 2026-11-15 | 11 | 2026-11-25 |
| `entsog_flujo_fisico` | incremental | 2026-11-16 | 154 | 2026-11-25 |
| `agsi_almacenamiento` | incremental | 2026-11-14 | 10 | 2026-11-23 |
| `miteco_embalses` | completa (primera carga) | 2014-01-01 | 245.421 | 2026-11-24 |

In [ ]:
%pip install access_parser

In [ ]:
import json, os, re, glob, datetime as dt
import requests
import pandas as pd

START = dt.date(2024, 1, 1)      # ventana de datos del proyecto
SOLAPE_DIAS = 7                  # días que se vuelven a pedir en cada carga
TIMEOUT = 60
HOY = dt.date.today()
TS = dt.datetime.now().strftime('%Y%m%dT%H%M%S')

ROOT = '/lakehouse/default/Files'
BRONZE = f'{ROOT}/bronze'
CFG_PATH = f'{ROOT}/config/utilities_config.json'

resultados = []

def ultima_fecha(fuente):
    """Mayor fecha guardada en Bronze para la fuente (lee el nombre del archivo), o None si es la primera carga."""
    fechas = []
    for f in glob.glob(f'{BRONZE}/{fuente}/*.json'):
        m = re.search(r'_hasta(\d{8})\.json$', f)
        if m:
            fechas.append(dt.datetime.strptime(m.group(1), '%Y%m%d').date())
    return max(fechas) if fechas else None

def desde_para(fuente):
    ult = ultima_fecha(fuente)
    if ult is None:
        return START, 'completa'
    return max(START, ult - dt.timedelta(days=SOLAPE_DIAS)), 'incremental'

def guardar(fuente, datos, max_fecha, desde, hasta, modo):
    """Escribe un JSON nuevo (nunca sobrescribe). Si no hay datos, no escribe nada."""
    if not datos:
        return None
    os.makedirs(f'{BRONZE}/{fuente}', exist_ok=True)
    nombre = f'{fuente}_{TS}_hasta{max_fecha.replace("-", "")}.json'
    ruta = f'{BRONZE}/{fuente}/{nombre}'
    meta = {'fuente': fuente, 'descargado': TS, 'modo': modo, 'desde': str(desde), 'hasta': str(hasta), 'max_fecha': max_fecha}
    with open(ruta, 'w', encoding='utf-8') as f:
        json.dump({'meta': meta, 'data': datos}, f, ensure_ascii=False)
    return ruta

def registrar(fuente, ok, modo='', desde=None, filas=None, max_fecha=None, nota=''):
    resultados.append({'fuente': fuente, 'ok': ok, 'modo': modo, 'desde': str(desde) if desde else None,
                       'filas': filas, 'max_fecha': max_fecha, 'nota': nota})

print(f'Hoy: {HOY} | START: {START} | solape: {SOLAPE_DIAS} días | ejecución: {TS}')

In [ ]:
# REE REData: demanda diaria. Se pide por trozos de como máximo un año.
FUENTE = 'ree_demanda'
try:
    desde, modo = desde_para(FUENTE)
    trozos, ini = [], desde
    while ini <= HOY:
        fin = min(dt.date(ini.year, 12, 31), HOY)
        r = requests.get('https://apidatos.ree.es/es/datos/demanda/evolucion',
                         params={'start_date': f'{ini}T00:00', 'end_date': f'{fin}T23:59', 'time_trunc': 'day'}, timeout=TIMEOUT)
        r.raise_for_status()
        trozos.append(r.json())
        ini = fin + dt.timedelta(days=1)

    fechas, filas = [], 0
    for j in trozos:
        for serie in j.get('included', []):
            for v in serie['attributes']['values']:
                fechas.append(v['datetime'][:10]); filas += 1
    max_f = max(fechas) if fechas else None
    ruta = guardar(FUENTE, trozos, max_f, desde, HOY, modo) if fechas else None
    registrar(FUENTE, True, modo, desde, filas, max_f, ruta or 'sin datos nuevos: no se escribe archivo')
except Exception as e:
    registrar(FUENTE, False, nota=str(e)[:200])

In [ ]:
# ENTSOG: flujo físico diario de Enagás, por meses desde la fecha de inicio hasta hoy.
FUENTE = 'entsog_flujo_fisico'
URL = 'https://transparency.entsog.eu/api/v1/operationaldatas'

def entsog_tramo(ini, fin):
    r = requests.get(URL, params={'indicator': 'Physical Flow', 'periodType': 'day', 'operatorLabel': 'Enagas',
                                  'from': ini, 'to': fin, 'limit': -1, 'timezone': 'CET'}, timeout=TIMEOUT)
    r.raise_for_status()
    j = r.json()
    clave = next((k for k, v in j.items() if isinstance(v, list)), None)
    return j[clave] if clave else []

try:
    desde, modo = desde_para(FUENTE)
    registros = []
    for ini in pd.date_range(desde.replace(day=1), HOY, freq='MS'):
        ini_d = max(ini.date(), desde)
        fin_d = min((ini + pd.offsets.MonthBegin(1)).date(), HOY)
        registros += entsog_tramo(ini_d.isoformat(), fin_d.isoformat())
    max_f = max(r['periodFrom'][:10] for r in registros) if registros else None
    ruta = guardar(FUENTE, registros, max_f, desde, HOY, modo) if registros else None
    registrar(FUENTE, True, modo, desde, len(registros), max_f, ruta or 'sin datos nuevos: no se escribe archivo')
except Exception as e:
    registrar(FUENTE, False, nota=str(e)[:200])

In [ ]:
# GIE AGSI+: almacenamiento subterráneo de España (requiere agsi_key)
FUENTE = 'agsi_almacenamiento'
if not os.path.exists(CFG_PATH):
    registrar(FUENTE, False, nota='omitida: falta utilities_config.json con agsi_key')
else:
    try:
        key = json.load(open(CFG_PATH))['agsi_key']
        desde, modo = desde_para(FUENTE)
        registros, page = [], 1
        while True:
            r = requests.get('https://agsi.gie.eu/api', params={'country': 'ES', 'from': desde.isoformat(), 'to': HOY.isoformat(),
                             'size': 300, 'page': page}, headers={'x-key': key}, timeout=TIMEOUT)
            r.raise_for_status()
            j = r.json()
            registros += j.get('data', [])
            if page >= int(j.get('last_page', 1)):
                break
            page += 1
        max_f = max(r['gasDayStart'] for r in registros) if registros else None
        ruta = guardar(FUENTE, registros, max_f, desde, HOY, modo) if registros else None
        registrar(FUENTE, True, modo, desde, len(registros), max_f, ruta or 'sin datos nuevos: no se escribe archivo')
    except Exception as e:
        registrar(FUENTE, False, nota=str(e)[:200])

In [ ]:
# MITECO: reserva de embalses peninsulares (semanal). El zip trae una base Access con el histórico desde 1988.
import io, zipfile, tempfile
from access_parser import AccessParser

FUENTE = 'miteco_embalses'
URL = 'https://www.miteco.gob.es/content/dam/miteco/es/agua/temas/evaluacion-de-los-recursos-hidricos/boletin-hidrologico/Historico-de-embalses/BD-Embalses.zip'
DESDE_AGUA = dt.date(2014, 1, 1)   # 10 años de historia para la media de 10 años
SOLAPE_AGUA = 56                   # MITECO revisa datos ("sin validación"): se vuelven a pedir 8 semanas

def col(cols, prefijo):
    return next(c for c in cols if c.upper().startswith(prefijo))

try:
    ult = ultima_fecha(FUENTE)
    desde, modo = (DESDE_AGUA, 'completa') if ult is None else (max(DESDE_AGUA, ult - dt.timedelta(days=SOLAPE_AGUA)), 'incremental')

    r = requests.get(URL, headers={'User-Agent': 'Mozilla/5.0'}, timeout=300)
    r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    nombre = next(n for n in z.namelist() if n.lower().endswith(('.mdb', '.accdb')))
    db = AccessParser(z.extract(nombre, tempfile.mkdtemp()))
    print('Tablas:', list(db.catalog.keys()))
    tabla = next(t for t in db.catalog if t.upper().startswith('T_DATOS EMBALSES'))
    t = db.parse_table(tabla)
    c = list(t.keys())
    cA, cE, cF = col(c, 'AMBITO'), col(c, 'EMBALSE'), col(c, 'FECHA')
    cT, cR, cL = col(c, 'AGUA_TOTA'), col(c, 'AGUA_ACTU'), col(c, 'ELECTRICO')

    df = pd.DataFrame({'ambito': t[cA], 'embalse': t[cE], 'fecha': pd.to_datetime(t[cF]),
                       'agua_total': t[cT], 'agua_actual': t[cR], 'electrico': t[cL]})
    df = df[df['fecha'].dt.date >= desde]
    df['fecha'] = df['fecha'].dt.strftime('%Y-%m-%d')
    registros = df.where(df.notna(), None).to_dict('records')
    max_f = df['fecha'].max() if len(df) else None
    ruta = guardar(FUENTE, registros, max_f, desde, HOY, modo) if registros else None
    registrar(FUENTE, True, modo, desde, len(registros), max_f, ruta or 'sin datos nuevos: no se escribe archivo')
except Exception as e:
    registrar(FUENTE, False, nota=str(e)[:200])

In [ ]:
res = pd.DataFrame(resultados)
display(res)
fallidas = res[~res['ok']]['fuente'].tolist()
if fallidas:
    # Hace que el pipeline marque la actividad como fallida y no ejecute Silver ni Gold
    raise RuntimeError(f'Fuentes con error u omitidas: {fallidas}')